# ICT4403 LLaMA English French
Kaggle run: https://www.kaggle.com/code/eltonlewis/ict4403-llama-english-french (version 3).
This notebook was assembled from the Kaggle MCP source and execution log. The code cell is the script executed by Kaggle; stream outputs are the saved run log. Checkpoints and data files are separate Kaggle output artifacts.
Saved output files: data/processed/manifest.json, data/processed/test.jsonl, data/processed/train.jsonl, data/processed/validation.jsonl, llama/metrics_greedy.json, llama/metrics_temp_0.7.json, llama/predictions_greedy.jsonl, llama/predictions_temp_0.7.jsonl.


In [1]:
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q","datasets>=3.0","transformers>=4.55,<5","accelerate>=1.0"])
"""Deterministic English-to-French OPUS Books preprocessing."""
import argparse
import hashlib
import json
import random
import re
import unicodedata
from pathlib import Path

DATASET = "Helsinki-NLP/opus_books"
CONFIG = "en-fr"
REVISION = "main"


def clean(value):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", value or "")).strip()


def prepare(rows, train_size=4000, val_size=200, test_size=200, seed=4403):
    """Deduplicate on source, then shuffle; source strings cannot cross splits."""
    unique = {}
    rejected = {"empty_or_length": 0, "length_ratio": 0, "duplicate_source": 0}
    for row in rows:
        pair = row["translation"]
        en, fr = clean(pair["en"]), clean(pair["fr"])
        if not (10 <= len(en) <= 300 and 10 <= len(fr) <= 350):
            rejected["empty_or_length"] += 1
            continue
        ratio = len(fr) / len(en)
        if not 0.45 <= ratio <= 2.2:
            rejected["length_ratio"] += 1
            continue
        key = en.casefold()
        if key in unique:
            rejected["duplicate_source"] += 1
            continue
        uid = hashlib.sha256((en + "\0" + fr).encode()).hexdigest()[:16]
        unique[key] = {"id": uid, "en": en, "fr": fr}
    pairs = list(unique.values())
    random.Random(seed).shuffle(pairs)
    n = train_size + val_size + test_size
    if len(pairs) < n:
        raise ValueError(f"Need {n} clean unique pairs; found {len(pairs)}")
    picked = pairs[:n]
    splits = {
        "train": picked[:train_size],
        "validation": picked[train_size:train_size + val_size],
        "test": picked[train_size + val_size:],
    }
    return splits, {"raw_rows": len(rows), "clean_unique_rows": len(pairs),
                    "rejected": rejected, "seed": seed,
                    "sizes": {k: len(v) for k, v in splits.items()},
                    "source": f"https://huggingface.co/datasets/{DATASET}/tree/{REVISION}/{CONFIG}",
                    "language_pair": "en-fr", "normalization": "Unicode NFKC and whitespace collapse",
                    "filters": "English 10-300 chars, French 10-350 chars, length ratio 0.45-2.2",
                    "deduplication": "casefolded English source; first occurrence retained"}


def write_splits(splits, manifest, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    for name, items in splits.items():
        with (output / f"{name}.jsonl").open("w", encoding="utf-8") as f:
            for item in items:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
    (output / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--output", default="data/processed")
    p.add_argument("--train-size", type=int, default=4000)
    p.add_argument("--val-size", type=int, default=200)
    p.add_argument("--test-size", type=int, default=200)
    p.add_argument("--seed", type=int, default=4403)
    a = p.parse_args()
    from datasets import load_dataset
    rows = load_dataset(DATASET, CONFIG, split="train", revision=REVISION)
    splits, manifest = prepare(rows, a.train_size, a.val_size, a.test_size, a.seed)
    write_splits(splits, manifest, a.output)
    print(json.dumps(manifest, indent=2))



sys.argv=['prepare_data']
main()
"""Few-shot, no-update LLaMA or Gemma translation on the fixed test split."""
import argparse
import json
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS = {"llama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
          "gemma": "unsloth/gemma-3-270m-it"}


def read(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]


def prompt(source, examples):
    shots = "\n".join(f"English: {r['en']}\nFrench: {r['fr']}" for r in examples)
    return ("Translate the final English text into French. Preserve names, numbers, "
            "and meaning. Return only the French translation.\n\n"
            f"Examples:\n{shots}\n\nEnglish: {source}\nFrench:")


def run(model_name, data_dir, output, temperature=0.0, sample_count=None):
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for this experiment")
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    train = read(Path(data_dir) / "train.jsonl")
    test = read(Path(data_dir) / "test.jsonl")
    if sample_count is not None:
        test = test[:sample_count]
    # Fixed examples come only from the training split and are shared by both models.
    examples = train[:3]
    model_id = MODELS[model_name]
    load_start = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16,
                                                device_map="auto", low_cpu_mem_usage=True).eval()
    load_seconds = time.perf_counter() - load_start
    predictions = []
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        for row in test:
            messages = [{"role": "user", "content": prompt(row["en"], examples)}]
            inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
            args = {"max_new_tokens": 96, "do_sample": temperature > 0,
                    "pad_token_id": tokenizer.eos_token_id}
            if temperature > 0:
                args.update(temperature=temperature, top_p=0.9)
            ids = model.generate(**inputs, **args)
            new_tokens = ids[0, inputs["input_ids"].shape[-1]:]
            prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            predictions.append(prediction)
    torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - start
    suffix = "greedy" if temperature == 0 else f"temp_{temperature:g}"
    with (output / f"predictions_{suffix}.jsonl").open("w", encoding="utf-8") as f:
        for row, prediction in zip(test, predictions):
            f.write(json.dumps({**row, "prediction": prediction}, ensure_ascii=False) + "\n")
    metrics = {"model": model_name, "model_id": model_id, "architecture": "decoder-only",
               "adaptation": "three-shot prompting; no parameter updates",
               "parameters": sum(p.numel() for p in model.parameters()),
               "training_seconds": 0, "load_seconds": load_seconds,
               "inference_seconds": inference_seconds, "test_count": len(test),
               "temperature": temperature, "max_new_tokens": 96,
               "device": torch.cuda.get_device_name(0),
               "few_shot_ids": [r["id"] for r in examples]}
    (output / f"metrics_{suffix}.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    return metrics



print(json.dumps(run('llama','data/processed','/kaggle/working/llama',0),indent=2),flush=True)
print(json.dumps(run('llama','data/processed','/kaggle/working/llama',0.7,30),indent=2),flush=True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.


gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


Generating train split: 100%|██████████| 127085/127085 [00:00<00:00, 553594.74 examples/s]


{


... (60 outputs left out)

/usr/local/lib/python3.13/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'


  text = re.sub(r'_', '\_', text) # Escape underscores in display text


[NbConvertApp] Converting notebook __script__.ipynb to html


[NbConvertApp] Writing 311725 bytes to __results__.html
